In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
# =====================================================
# 1. LOAD DATASET
# =====================================================

file_path = "dataset.csv"   # change dataset name

df = pd.read_csv(file_path)

print("Dataset Loaded")
print(df.head())

In [ ]:
# =====================================================
# 2. HANDLE MISSING VALUES
# =====================================================

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].mean())

for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# =====================================================
# 3. ENCODE CATEGORICAL COLUMNS
# =====================================================

encoder = LabelEncoder()

for col in df.select_dtypes(exclude=np.number).columns:
    df[col] = encoder.fit_transform(df[col])

In [ ]:
# =====================================================
# 4. SELECT TARGET COLUMN
# =====================================================

target_column = df.columns[-1]   # last column as target

X = df.drop(columns=[target_column])
y = df[target_column]

In [ ]:
# =====================================================
# 5. NORMALIZATION
# =====================================================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# =====================================================
# 6. VERTICAL FEATURE SPLIT
# =====================================================

num_features = X_scaled.shape[1]
mid = num_features // 2

XA = X_scaled[:, :mid]
XB = X_scaled[:, mid:]

In [ ]:
# =====================================================
# 7. TRAIN TEST SPLIT
# =====================================================

XA_train, XA_test, XB_train, XB_test, y_train, y_test = train_test_split(
    XA,
    XB,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# =====================================================
# 8. CONVERT TO TENSORS
# =====================================================

XA_train = torch.tensor(XA_train, dtype=torch.float32)
XB_train = torch.tensor(XB_train, dtype=torch.float32)

XA_test = torch.tensor(XA_test, dtype=torch.float32)
XB_test = torch.tensor(XB_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values.reshape(-1,1), dtype=torch.float32)
y_test = torch.tensor(y_test.values.reshape(-1,1), dtype=torch.float32)

In [ ]:
# =====================================================
# 9. PARTY MODELS
# =====================================================

class PartyA(nn.Module):

    def __init__(self, input_size):
        super().__init__()
        self.fc = nn.Linear(input_size, 4)

    def forward(self, x):
        return torch.relu(self.fc(x))


class PartyB(nn.Module):

    def __init__(self, input_size):
        super().__init__()
        self.fc = nn.Linear(input_size, 4)

    def forward(self, x):
        return torch.relu(self.fc(x))


In [ ]:
# =====================================================
# 10. AGGREGATOR
# =====================================================

class Aggregator(nn.Module):

    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(8, 1)

    def forward(self, a_out, b_out):

        combined = torch.cat([a_out, b_out], dim=1)

        return torch.sigmoid(self.fc(combined))

In [ ]:
# =====================================================
# 11. INITIALIZE MODELS
# =====================================================

partyA = PartyA(XA_train.shape[1])
partyB = PartyB(XB_train.shape[1])

aggregator = Aggregator()


In [ ]:
# =====================================================
# 12. LOSS & OPTIMIZER
# =====================================================

criterion = nn.BCELoss()

optimizerA = optim.Adam(partyA.parameters(), lr=0.01)
optimizerB = optim.Adam(partyB.parameters(), lr=0.01)
optimizerAgg = optim.Adam(aggregator.parameters(), lr=0.01)

In [ ]:
# =====================================================
# 13. TRAINING LOOP
# =====================================================

epochs = 50

for epoch in range(epochs):

    optimizerA.zero_grad()
    optimizerB.zero_grad()
    optimizerAgg.zero_grad()

    # Local computations
    a_out = partyA(XA_train)
    b_out = partyB(XB_train)

    # Aggregation
    predictions = aggregator(a_out, b_out)

    # Loss
    loss = criterion(predictions, y_train)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizerA.step()
    optimizerB.step()
    optimizerAgg.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


In [ ]:
# =====================================================
# 14. TESTING
# =====================================================

with torch.no_grad():

    a_out = partyA(XA_test)
    b_out = partyB(XB_test)

    preds = aggregator(a_out, b_out)

    preds = (preds > 0.5).float()

    accuracy = (preds == y_test).float().mean()

    print("\nFinal Accuracy:", accuracy.item())